# ogbn-arxiv subgraph extraction for GraphSAGE

Extracts a manageable k-hop subgraph (~20k–40k nodes) from the full ogbn-arxiv graph,
attaches the correct 128-d node embeddings, and produces train/val/test masks
derived from the original OGB time-based splits.

In [1]:
import json

import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
from torch_geometric.utils import k_hop_subgraph, to_undirected

/opt/anaconda3/envs/DyTAG/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load raw data

In [2]:
DATA_DIR = "../ogbn_arxiv_dataset"

nodes_df = pd.read_csv(f"{DATA_DIR}/nodes.csv")          # node_id, mag_id, title, abstract, label, year
edges_df = pd.read_csv(f"{DATA_DIR}/edges.csv")          # src_node_id, dst_node_id
node_features_np = np.load(f"{DATA_DIR}/node_features.npy")  # shape (169343, 128)

with open(f"{DATA_DIR}/splits.json") as f:
    splits = json.load(f)

num_nodes = len(nodes_df)  # 169 343
print(f"Nodes        : {num_nodes:,}")
print(f"Edges        : {len(edges_df):,}")
print(f"Feature dim  : {node_features_np.shape[1]}")
print(f"Classes      : {nodes_df['label'].nunique()}")
print(f"Train / Val / Test : {len(splits['train']):,} / {len(splits['valid']):,} / {len(splits['test']):,}")

Nodes        : 169,343
Edges        : 1,166,243
Feature dim  : 128
Classes      : 40
Train / Val / Test : 90,941 / 29,799 / 48,603


## 2. Build full PyG graph

node_id is contiguous (0 … 169 342) and equals the row index, so
`node_features_np[node_id]` is always the correct embedding for a node.

In [3]:
# Edge index in COO format required by PyG
edge_index_directed = torch.tensor(
    edges_df[["src_node_id", "dst_node_id"]].values.T,
    dtype=torch.long,
)

# Make undirected: GraphSAGE aggregates from all neighbours,
# so reverse edges are needed to propagate information both ways.
edge_index = to_undirected(edge_index_directed, num_nodes=num_nodes)

x_full = torch.from_numpy(node_features_np).float()           # (169343, 128)
y_full = torch.tensor(nodes_df["label"].values, dtype=torch.long)  # (169343,)

print(f"Directed edges  : {edge_index_directed.shape[1]:,}")
print(f"Undirected edges: {edge_index.shape[1]:,}  (both directions)")

Directed edges  : 1,166,243
Undirected edges: 2,315,598  (both directions)


## 3. Select seed nodes and extract k-hop subgraph

Strategy: seed with a random mix of train + test nodes, then expand 1 hop.
Tune `SEED_SIZE` to control the final subgraph size (~20k–40k nodes).
With 3 000 seeds on this graph a 1-hop expansion yields roughly 30k–38k nodes.

In [ ]:
SEED_SIZE = 3000   
NUM_HOPS  = 1
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)

# Draw seeds proportionally from train / val / test so all splits are represented
train_ids = np.array(splits["train"])
valid_ids  = np.array(splits["valid"])
test_ids   = np.array(splits["test"])

n_train_seed = int(SEED_SIZE * 0.6)
n_valid_seed = int(SEED_SIZE * 0.2)
n_test_seed  = SEED_SIZE - n_train_seed - n_valid_seed

seed_ids = np.concatenate([
    rng.choice(train_ids, size=n_train_seed, replace=False),
    rng.choice(valid_ids,  size=n_valid_seed,  replace=False),
    rng.choice(test_ids,  size=n_test_seed,  replace=False),
])

seed_nodes = torch.tensor(seed_ids, dtype=torch.long)
print(f"Seed nodes: {len(seed_nodes):,}  (train {n_train_seed} / val {n_valid_seed} / test {n_test_seed})")

Seed nodes: 3,000  (train 1800 / val 600 / test 600)


In [11]:
subset, edge_index_sub, mapping, edge_mask = k_hop_subgraph(
    node_idx=seed_nodes,
    num_hops=NUM_HOPS,
    edge_index=edge_index,
    relabel_nodes=True,   # renumbers nodes 0 … n_sub-1
    num_nodes=num_nodes,
)

print(f"Subgraph nodes : {len(subset):,}")
print(f"Subgraph edges : {edge_index_sub.shape[1]:,}")
print(f"subset dtype   : {subset.dtype}  — original node_ids of each subgraph node")

Subgraph nodes : 36,150
Subgraph edges : 744,062
subset dtype   : torch.int64  — original node_ids of each subgraph node


## 4. Index features and labels correctly

`subset[i]` holds the **original** node_id of the i-th subgraph node.
Because `node_id == row_index` in the dataset, indexing `x_full` and `y_full`
with `subset` gives the correct embeddings and labels — no mapping table needed.

In [12]:
x_sub = x_full[subset]   # (n_sub, 128)  — correct embedding per subgraph node
y_sub = y_full[subset]   # (n_sub,)      — correct label per subgraph node

print(f"x_sub : {tuple(x_sub.shape)}  dtype={x_sub.dtype}")
print(f"y_sub : {tuple(y_sub.shape)}  dtype={y_sub.dtype}")
print(f"Label classes present: {y_sub.unique().numel()}")

x_sub : (36150, 128)  dtype=torch.float32
y_sub : (36150,)  dtype=torch.int64
Label classes present: 40


## 5. Train / val / test masks

A subgraph node's split membership is determined by the original OGB split,
not by whether it was a seed node.

In [13]:
train_set = set(splits["train"])
valid_set  = set(splits["valid"])
test_set  = set(splits["test"])

subset_np = subset.numpy()

train_mask = torch.tensor([int(n) in train_set for n in subset_np], dtype=torch.bool)
val_mask   = torch.tensor([int(n) in valid_set  for n in subset_np], dtype=torch.bool)
test_mask  = torch.tensor([int(n) in test_set  for n in subset_np], dtype=torch.bool)

print(f"Train nodes in subgraph : {train_mask.sum().item():,}")
print(f"Val   nodes in subgraph : {val_mask.sum().item():,}")
print(f"Test  nodes in subgraph : {test_mask.sum().item():,}")
print(f"(unlabelled / overlap)  : {(~train_mask & ~val_mask & ~test_mask).sum().item():,}")

Train nodes in subgraph : 17,198
Val   nodes in subgraph : 6,929
Test  nodes in subgraph : 12,023
(unlabelled / overlap)  : 0


## 6. Assemble PyG Data object

In [14]:
data = Data(
    x=x_sub,
    edge_index=edge_index_sub,
    y=y_sub,
    train_mask=train_mask,
    val_mask=val_mask,
    test_mask=test_mask,
    original_node_ids=subset,   # original ogbn-arxiv node_ids, useful for lookup
)

print(data)
print(f"\nNum nodes  : {data.num_nodes:,}")
print(f"Num edges  : {data.num_edges:,}")
print(f"Num classes: {int(data.y.max()) + 1}")

Data(x=[36150, 128], edge_index=[2, 744062], y=[36150], train_mask=[36150], val_mask=[36150], test_mask=[36150], original_node_ids=[36150])

Num nodes  : 36,150
Num edges  : 744,062
Num classes: 40


## 7. Save subgraph for reuse

In [15]:
torch.save(data, "subgraph_data.pt")
print("Saved to GraphSAGE_classification/subgraph_data.pt")

# Load back with: data = torch.load("subgraph_data.pt")

Saved to GraphSAGE_classification/subgraph_data.pt
